In [ ]:
import os
from statistics import mean 
import pandas as pd  
        
def getParetoFrontPerRun(file_path, whichPareto):
    acc_list, recall_list, precision_list, f1_list, mcc_list, spd_list, aod_list, eod_list= [],[],[],[],[],[],[],[]
    with open(file_path, 'r') as f:                                             
            my_scores = []
            for idx, ln in enumerate(f):
                if ln.startswith('score'):
                    my_scores.append([ln])
                        
            for single_score in my_scores:
                str_split = single_score[0].replace("score:{", "")
                str_split = str_split.replace("(", " ")
                str_split = str_split.replace(")", " ")
                acc_comp = "\'" + 'accuracy' + "\': "
                recall_comp = "\'" + 'recall' + "\': "
                precision_comp = "\'" + 'precision' + "\': "
                f1_comp = "\'" + 'f1' + "\': "
                mcc_comp = "\'" + 'mcc' + "\': np.float64" if "'mcc': np.float64" in str_split else "\'" + 'mcc' + "\': "
                spd_comp = "\'" + 'spd' + "\': np.float64" if "'spd': np.float64" in str_split else "\'" + 'spd' + "\': "
                aod_comp = "\'" + 'aod' + "\': np.float64" if "'aod': np.float64" in str_split else "\'" + 'aod' + "\': "
                eod_comp = "\'" + 'eod' + "\': np.float64" if "'eod': np.float64" in str_split else "\'" + 'eod' + "\': "
                avg_spd = "\'" + 'avg_spd' + "\': np.float64" if "'avg_spd': np.float64" in str_split else "\'" + 'avg_spd' + "\': "
                avg_aod = "\'" + 'avg_aod' + "\': np.float64" if "'avg_aod': np.float64" in str_split else "\'" + 'avg_aod' + "\': "
                avg_eod = "\'" + 'avg_eod' + "\': np.float64" if "'avg_eod': np.float64" in str_split else "\'" + 'avg_eod' + "\': "

                
                
                acc_list.append(float(str_split.split(acc_comp)[1].split(',')[0]))
                recall_list.append(float(str_split.split(recall_comp)[1].split(',')[0]))
                precision_list.append(float(str_split.split(precision_comp)[1].split(',')[0]))
                f1_list.append(float(str_split.split(f1_comp)[1].split(',')[0]))
                mcc_list.append(float(str_split.split(mcc_comp)[1].split(',')[0]))
                spd_list.append(float(str_split.split(spd_comp)[1].split(',')[0]))
                aod_list.append(float(str_split.split(aod_comp)[1].split(',')[0]))
                eod_list.append(float(str_split.split(eod_comp)[1].split(',')[0].replace('}\n', '')))
    
    if whichPareto == 'max':
        return max(acc_list), max(recall_list), max(precision_list), max(f1_list), max(mcc_list), max(spd_list), max(aod_list), max(eod_list)
    elif whichPareto == 'avg': 
        return mean(acc_list), mean(recall_list), mean(precision_list), mean(f1_list), mean(mcc_list), mean(spd_list), mean(aod_list), mean(eod_list)
    elif whichPareto == 'min':
            return min(acc_list), min(recall_list), min(precision_list), min(f1_list), min(mcc_list), min(spd_list), min(aod_list), min(eod_list)     
    return acc_list, recall_list, precision_list, f1_list, mcc_list, spd_list, aod_list, eod_list

def extractResultsAllRuns(folder_path):
    acc_runs_list, recall_runs_list, precision_runs_list, f1_runs_list, mcc_runs_list, spd_runs_list, aod_runs_list, eod_runs_list =  [],[],[],[],[],[],[],[]     
    results_df = pd.DataFrame()
    for file in os.listdir(folder_path):
        if not file.endswith('.png') and not os.path.isdir(os.path.join(folder_path, file)) and not file.startswith('.'):
            round_num = file.split('_')[-1].replace('.txt', '')
            file_path = folder_path +'/'+ file
            print('printing file path: ', file_path)
            acc_value, recall_value, precision_value, f1_value, mcc_value, spd_value, aod_value, eod_value = getParetoFrontPerRun(
                file_path, 'full')         
            
            acc_runs_list.append(acc_value)
            recall_runs_list.append(recall_value)
            precision_runs_list.append(precision_value)
            f1_runs_list.append(f1_value)
            mcc_runs_list.append(mcc_value)
            spd_runs_list.append(spd_value)
            aod_runs_list.append(aod_value)
            eod_runs_list.append(eod_value)

            round_results = pd.DataFrame({
                 'Accuracy': acc_value, 'Recall': recall_value, 'Precision': precision_value,
                 'F1': f1_value, 'MCC': mcc_value, 'SPD': spd_value,
                 'AOD': aod_value, 'EOD': eod_value
            })
            round_results['Run'] = round_num
            results_df = pd.concat([results_df, round_results], ignore_index=True)

            # results_df = pd.DataFrame({'Accuracy': acc_runs_list, 'Recall': recall_runs_list,
            #     'Precision': precision_runs_list, 'F1': f1_runs_list,
            #     'MCC': mcc_runs_list, 'SPD': spd_runs_list,
            #     'AOD': aod_runs_list, 'EOD': eod_runs_list})
     
    return results_df


rootdir = '../mut_rf/outputs_svm'
       
for folder in os.listdir(rootdir):
    folder_path = os.path.join(rootdir, folder)
    if folder.endswith('.ipynb') or folder.startswith('.DS') or folder.endswith('.csv'):
        continue
    else: 
        attr = folder.split('_')[1]
        file = folder.split('_')[0]

        # if folder == 'Adult' or folder == 'Compas':
        #     for subfolder in os.listdir(folder):
        #         if not subfolder.startswith('.'):
        #             attr = subfolder
        #             folder_path = os.path.join(rootdir, folder, subfolder)
        #             results_df = extractResultsAllRuns(folder_path)
        #             results_file_name = folder + '_' + attr + '_results.csv'  
        #             results_df.to_csv(results_file_name)  
        # else:
        results_df = extractResultsAllRuns(folder_path)
        results_file_name = file + '_' + attr + '_results.csv'
        os.makedirs('results_svm', exist_ok=True)  # Ensure the directory exists
        results_df.to_csv(f"results_svm/{results_file_name}")



printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_7.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_17.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_16.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_6.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_4.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_14.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_15.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_5.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_1.txt
printing file path:  mut_no_ensemble/outputs_svm/adult_sex/adult_testset_sex_accuracy+spd_run_11